# w9_a100.ipynb — R60 wcle protocol on A100 (80 GB VRAM / 200 GB RAM)

One ARM per GPU job: tower 1000ep (ckpt/50) + per-checkpoint heads (3 seeds)
+ post-hoc vsel pick topped to 10 seeds. The ENTIRE corpus lives in VRAM
(pool 8.5 GB + flat pseudo-queries 8.5 GB + anchors + views ~= 21 GB), view
sampling gathers on-GPU — no host copies in the training loop.

**Upload once to `/workspace/fusion_cache_w9`** (from the local scratchpad
`fusion_cache`): games.npz, wiki_eval.npz, wscan_gal_rev.npz,
wscan_pool_rev.npy, wscan_pool_rev_rid.npy, wscan_pool_rev_len.npy,
ss_queries_rev.npz, ss_queries_rev_S.npy, wiki_clean_views.npz,
sp_raw_views.npz, tag_labels.npz, wiki_eval_split.json, _tag_splitM.json
(~28 GB total).

Jobs below: the four arms remaining from the local 3080 queue @512 anchors,
plus the champion recipe (cegate2) re-run with the FULL 2048-sentence anchor
— the A100-only experiment. Edit JOBS to taste; done jobs auto-skip.

In [ ]:
# w9_a100.ipynb -- all pod-specific constants live HERE.
import os, subprocess

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"    # RAM staging (200 GB RAM; corpus ~28 GB)
OUT_DIR = "/workspace/w9_out"

# (arm, anchor_cap) jobs. 512 = local-protocol continuation; 2048 = full-anchor
# A100 showcase (the user's original anchor budget, unaffordable on the 3080).
# (arm, anchor_cap, no_sp_view). nsp=True = WIKI-PURE tower views: doc tier is
# wiki_clean only (407 games), no sp_raw fallback -- the sp-ablation of every
# tested recipe (user decree 2026-07-12). Anchors/pseudo-queries unchanged.
JOBS = [
    ("wcle_cegate3_icetf", 512, False),   # champion recipe, I x3 (dose curve pt 4)
    ("wcle_cegate4_icetf", 512, False),   # champion recipe, I x4 (dose curve pt 5)
    ("wcle_cegate2_icetf", 2048, False),  # champion recipe x FULL 2048 anchor
    ("wcle_cegate2_icetf", 512, True),    # nsp roster: every tested recipe, wiki-pure
    ("wcle_cegate1_icetf", 512, True),
    ("wcle_i2ce_icetf", 512, True),
    ("wcle_ice_icetf", 512, True),
    ("wcle_ce_cetf", 512, True),
    ("wcle_arc_arctf", 512, True),
    ("wcle_byol_bytf", 512, True),
]
# (igate1/igate1w dropped 2026-07-12: local grid proved CE-gating is the lever;
#  cegate1w/2w have no nsp variant -- with sp views gone they coincide with
#  cegate1/2 nsp.)
EPOCHS, CKPT_EVERY, CKPT_SEEDS, TOPUP_SEEDS = 1000, 50, 3, 10

def _detect_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=5).stdout.strip()
        ids = [l.strip() for l in out.splitlines() if l.strip()]
        return ids if ids else ["0"]
    except Exception:
        return ["0"]

GPUS = _detect_gpus()
os.makedirs(OUT_DIR, exist_ok=True)
print("repo :", REPO)
print("data :", DATA_SRC, "->", DATA_RAM)
print("out  :", OUT_DIR)
print("jobs :", [f"{a}@g{c}" for a, c in JOBS])
print("gpus :", GPUS, "(one worker per GPU)")

In [ ]:
# Clone or FORCE-sync to origin/main before every run. The pod repo is a MIRROR
# of GitHub: reset --hard discards pod-local edits in tracked paths.
import os, importlib.util

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD

for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break

In [ ]:
# Verify the corpus and stage it into RAM (/dev/shm): one shared copy; the
# workers then lift what they need into their own GPU's VRAM.
import shutil
from pathlib import Path

REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing} -- upload first"

dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
total = sum((dst / f).stat().st_size for f in REQUIRED) / 1e9
print(f"corpus in RAM: {DATA_DIR} ({total:.1f} GB)")

In [ ]:
# Run all (arm, anchor_cap) jobs across the GPUs: one subprocess per GPU,
# pinned via CUDA_VISIBLE_DEVICES, pulling from a shared queue. Resume by
# default (per-checkpoint caches + per-ft json). Logs in OUT_DIR/logs/.
import os, queue, subprocess, threading, time
from pathlib import Path

jobs = queue.Queue()
n_jobs = 0
for arm, cap, nsp in JOBS:
    nm = f"w9_{arm}" + (f"_g{cap}" if cap != 512 else "") + ("_nsp" if nsp else "")
    if (Path(OUT_DIR) / f"ft4var_{nm}_best.json").exists():
        print(f"[skip] {nm} already done")
        continue
    jobs.put((arm, cap, nsp)); n_jobs += 1
log_dir = Path(OUT_DIR) / "logs"
log_dir.mkdir(exist_ok=True)
fails = []

def worker(gpu):
    while True:
        try:
            arm, cap, nsp = jobs.get_nowait()
        except queue.Empty:
            return
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
        log = log_dir / f"{arm}_g{cap}{'_nsp' if nsp else ''}.log"
        print(f"[gpu{gpu}] start {arm}@g{cap}{'/nsp' if nsp else ''} -> logs/{log.name}", flush=True)
        t0 = time.time()
        cmd = ["python", "-u", os.path.join(REPO, "Pod/w9_a100_worker.py"),
               "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--repo", REPO,
               "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(EPOCHS), "--ckpt-every", str(CKPT_EVERY),
               "--ckpt-seeds", str(CKPT_SEEDS), "--topup-seeds", str(TOPUP_SEEDS)]
        if nsp:
            cmd.append("--no-sp-view")
        with open(log, "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env)
        status = "ok" if p.returncode == 0 else f"FAIL rc={p.returncode}"
        if p.returncode != 0:
            fails.append((arm, cap, str(log)))
        print(f"[gpu{gpu}] {status} {arm}@g{cap} [{(time.time()-t0)/60:.1f} min]", flush=True)

threads = [threading.Thread(target=worker, args=(g,)) for g in GPUS]
t0 = time.time()
for t in threads: t.start()
for t in threads: t.join()
print(f"
finished in {(time.time()-t0)/60:.1f} min; {n_jobs} run, {len(fails)} failed")
for arm, cap, log in fails:
    print("  FAILED:", arm, f"g{cap}", "-> check", log)

In [ ]:
# Aggregate: per-arm best tables (vsel-picked checkpoint, 10 seeds).
import json
import numpy as np
from pathlib import Path

VORD = ["neutral", "noname", "positive", "negative"]
for j in sorted(Path(OUT_DIR).glob("ft4var_*_best*.json")):
    d = json.loads(j.read_text())
    runs = d["per_seed"]
    row = {v: (np.mean([r[v]["h1"] for r in runs]),
               np.mean([r[v]["h5"] for r in runs]),
               np.mean([r[v]["tag"] for r in runs])) for v in VORD}
    m4 = np.mean([np.mean([r[v]["h1"] for r in runs]) for v in VORD])
    print(f"{j.stem} (ep{d.get('best_ep','?')}, n={len(runs)})")
    for v in VORD:
        h1, h5, tg = row[v]
        print(f"  {v:9s} h1={h1:.3f} h5={h5:.3f} tag={tg:.3f}")
    print(f"  mean-of-4 = {m4:.3f}")